# SafeStack — Phase 5 FU3b: robustness-stress continue-train on Colab (A100)

Continue-trains the pinned SFT LoRA adapter (C5 = budget 0) on the five **robustness-stress** budget
slices (ADR-0017 dec.3), producing five **stressed** adapters, and uploads each to a **private** HF-Hub
repo for an immutable id (**ADR-0017 Amendment 2** permits the private hub; the commit SHA is the pin,
ADR-0015 dec.7b). The bare stressed policy at the dev-selected budget b\* is the **C9** condition;
**C10** adds the guardrails. **b\* selection and the C9/C10 eval are FU4 — not here.**

Each budget is a continue-train from the SAME pinned SFT adapter
(`kambleakash0/safestack-sft-mistral-lora-v1 @ 05266a9b…`) on its `stress_sorrybench_v1_b{budget}` slice,
holding the SFT recipe identical so only the dose (N stress examples, each seen once) varies (dec.3).

**Committed (aggregate-only):** the five loss curves and this executed notebook. **Private, never
committed, never public:** the five stressed adapter weights live in **private** HF-Hub repos + gitignored
`adapters/` (ADR-0017 dec.7 as amended by **Amendment 2** — private, never made public), and the raw
stress prompts + affirmative-onset targets stay gitignored.

Runtime → Change runtime type → **GPU (A100)**. Needs Colab secrets `HF_TOKEN` (gated SORRY-Bench +
WildJailbreak + the private SFT adapter to resume + the private stressed repos to create) and `GH_TOKEN`
(clone the private repo).

In [1]:
# 1. GPU check
import platform

import torch

print("python :", platform.python_version())
print("torch  :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU    :", props.name)
    print("VRAM   :", round(props.total_memory / 1e9, 1), "GB")
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU (A100).")

python : 3.12.13
torch  : 2.11.0+cu128 | CUDA available: True
GPU    : NVIDIA A100-SXM4-40GB
VRAM   : 42.4 GB


In [2]:
# 2. Secrets + clone/update the (private) repo
import os
import stat
import subprocess
import tempfile

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]  # transformers/datasets read this
os.environ["GIT_TOKEN"] = userdata.get("GH_TOKEN")            # token stays in the ENV, never in argv
REPO = "kambleakash0/safestack-study"
DEST = "/content/safestack-study"
REPO_URL = f"https://github.com/{REPO}.git"                   # tokenless remote (no PAT in .git/config)

# Auth via a GIT_ASKPASS helper that READS the token from the environment: the script itself holds no
# secret, and the token reaches git through the (owner-only) process env, not a command-line argument
# (argv is world-readable via /proc/<pid>/cmdline), and never touches .git/config or disk.
_askpass = tempfile.NamedTemporaryFile("w", suffix=".sh", delete=False)
_askpass.write('#!/bin/sh\ncase "$1" in *[Uu]sername*) echo x-access-token ;; *) echo "$GIT_TOKEN" ;; esac\n')
_askpass.close()
os.chmod(_askpass.name, stat.S_IRWXU)                        # 0700, owner-only
_git_env = {**os.environ, "GIT_ASKPASS": _askpass.name, "GIT_TERMINAL_PROMPT": "0"}

# Clone if missing, else force the checkout to the latest main. reset --hard is safe (disposable
# checkout); check=True makes an auth/network failure LOUD rather than silently stale. try/finally so
# the token and the askpass helper are ALWAYS cleaned up -- even if a git op raises, the GH_TOKEN
# never lingers in the kernel env and no helper file is left on disk.
try:
    if not os.path.isdir(DEST):
        subprocess.run(["git", "clone", "-q", REPO_URL, DEST], check=True, env=_git_env)
    subprocess.run(["git", "-C", DEST, "fetch", "-q", "origin", "main"], check=True, env=_git_env)
    subprocess.run(["git", "-C", DEST, "reset", "--hard", "-q", "origin/main"], check=True, env=_git_env)
finally:
    os.remove(_askpass.name)            # drop the askpass helper (even on failure)
    os.environ.pop("GIT_TOKEN", None)   # drop the token from the environment (even on failure)
%cd /content/safestack-study
!git log --oneline -1

/content/safestack-study
31036d5 (HEAD -> main, origin/main, origin/HEAD) feat(nb): FU3b stress-train Colab notebook + ADR-0017 Amendment 2 (private-hub storage) + logging_steps fix (#121)


In [3]:
# 3. Install SafeStack + the [train] extra (LoRA/QLoRA: bitsandbytes + accelerate; peft via [hf])
!pip -q install -e ".[train]"
# Colab preinstalls torchao 0.10.0, which the newer PEFT rejects (needs > 0.16.0) and RAISES on when
# loading a LoRA adapter onto a non-4bit (bf16) base. We use bitsandbytes, not torchao, so remove it:
# PEFT's is_torchao_available() then returns False and skips that dispatcher cleanly. (The 4-bit
# continue-train path short-circuits to the bnb dispatcher first, but keep the setup identical to FU5c.)
!pip -q uninstall -y torchao
import peft
import transformers

print("transformers", transformers.__version__, "| peft", peft.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 50.0 MB/s eta 0:00:00
  Building editable for safestack (pyproject.toml) ... done
transformers 5.13.1 | peft 0.19.1


In [4]:
# 4. Mount Drive for resumable caches + adapter staging (a killed session resumes in minutes)
from google.colab import drive

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/safestack"
CACHE = f"{BASE}/cache"
RUNS = f"{BASE}/runs"
ADAPTERS = f"{BASE}/adapters"          # persistent copies of the stressed adapters (also on HF-Hub)
REPORTS = "/content/safestack-study/reports"
BUDGETS = [10, 50, 100, 250, 411]      # the dose-response grid (ADR-0017 dec.3 + Amendment 1)
# One config + one output adapter per budget; ADAPTER_LOCAL[b] matches output_adapter in each config.
ADAPTER_LOCAL = {b: f"adapters/stress_mistral_lora_b{b}" for b in BUDGETS}
STRESS_CONFIG = {b: f"configs/train/stress_mistral_lora_b{b}.yaml" for b in BUDGETS}
for d in (CACHE, RUNS, ADAPTERS, REPORTS):
    os.makedirs(d, exist_ok=True)
print("budgets:", BUDGETS)
print("cache  :", CACHE)

Mounted at /content/drive
budgets: [10, 50, 100, 250, 411]
cache  : /content/drive/MyDrive/safestack/cache


In [5]:
# 5. Prepare the reference suites in dependency order, then the stress slices. The prep guards FAIL
#    CLOSED on a partial set: DEV prep needs eval + train_sft prepared, and prepare-stress needs eval +
#    dev prepared -- so the order is eval -> train_sft -> dev -> stress (the DAG the #119 fix restored).
#    WildJailbreak (train_sft) and SORRY-Bench (stress) are gated -> need the HF token. This regenerates
#    the 411-disjoint stress slices (deterministic, seed 0). A prepare failure STOPS here.
def _prep(cmd, label):
    print(f"--- {label} ---")
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-1500:], end="")
    if p.returncode != 0:
        print(p.stderr[-2000:])
        raise SystemExit(f"prepare failed: {label}")

EVAL_SUITES = [
    "harmful_advbench_v1",
    "harmful_harmbench_v1",
    "dualuse_harmbench_contextual_v1",   # eval_dual_use: ADR-0015's top-priority leakage-dedup target
    "overrefusal_xstest_v1",
    "helpfulness_alpaca_v1",
]
DEV_SUITES = [
    "dev_harmful_maliciousinstruct_v1",
    "dev_overrefusal_orbench_v1",
    "dev_helpfulness_alpaca_v1",
]
for name in EVAL_SUITES:
    _prep(["safestack", "data", "prepare", "-c", f"configs/datasets/{name}.yaml"], name)
_prep(["safestack", "data", "prepare-sft", "-c", "configs/datasets/sft_wildjailbreak_v1.yaml"],
      "train_sft (WildJailbreak, gated)")
for name in DEV_SUITES:
    _prep(["safestack", "data", "prepare", "-c", f"configs/datasets/{name}.yaml"], name)
_prep(["safestack", "data", "prepare-stress", "-c", "configs/datasets/stress_sorrybench_v1.yaml"],
      "train_robustness_stress (SORRY-Bench, gated)")

--- harmful_advbench_v1 ---
prepared harmful_advbench_v1: 520 records -> sha256:a80ecfba71fadd12f194a658b924cbf6dd6b014f6b1d93057db4f422e2cfb4c3
--- harmful_harmbench_v1 ---
prepared harmful_harmbench_v1: 200 records -> sha256:1aabe6806d144c5d86ac03d64c77a6ba76f9446c0dc98833d300f24959f4b82f
--- dualuse_harmbench_contextual_v1 ---
prepared dualuse_harmbench_contextual_v1: 100 records -> sha256:52ced8ea6b4a8df5da1ad95793a00568ab517acb8a621df4e16f69dfede18ef7
--- overrefusal_xstest_v1 ---
prepared overrefusal_xstest_v1: 250 records -> sha256:24bd1fad943d9a368632b4b97d6d7f52aabda05a757c03c4dc8c87d3f6928fb6
--- helpfulness_alpaca_v1 ---
prepared helpfulness_alpaca_v1: 200 records -> sha256:31d0aa39d2f6d31294ee86a8b4829b24483434c6edcf8c01236ff30b93444d66
--- train_sft (WildJailbreak, gated) ---
prepared sft_wildjailbreak_v1: 10000 records -> sha256:34018e6c1356fd007fcaac2138ce6d9d288c566c1f8037e2b79fe2958c95a4fc
--- dev_harmful_maliciousinstruct_v1 ---
prepared dev_harmful_maliciousinstruct_

In [6]:
# 6. Drift guard (content-hash only). The committed manifests pin each source's content hash; cell 5
#    just regenerated them. Compare ONLY the `hash` field (created_at is restamped every prep, so a
#    whole-file diff would false-positive). A real upstream drift (a pinned source moved, or a parsing
#    shift) changes the hash -> STOP before training. Covers eval + train_sft + dev AND the 5 slices.
import yaml

_stress_manifests = [f"stress_sorrybench_v1_b{b}" for b in BUDGETS]
_manifests = EVAL_SUITES + ["sft_wildjailbreak_v1"] + DEV_SUITES + _stress_manifests
_drift = []
for _name in _manifests:
    _path = f"data/manifests/{_name}.yaml"
    _regen = yaml.safe_load(open(_path))["hash"]
    _committed = yaml.safe_load(
        subprocess.run(["git", "show", f"HEAD:{_path}"], capture_output=True, text=True).stdout
    )["hash"]
    if _regen != _committed:
        _drift.append(f"{_name}: committed {_committed} != regenerated {_regen}")
if _drift:
    print("\n".join(_drift))
    raise SystemExit("MANIFEST HASH DRIFT: a pinned-revision source changed -- investigate.")
print("no data drift: all", len(_manifests), "manifest content hashes match the committed pins")

no data drift: all 14 manifest content hashes match the committed pins


In [7]:
# 7. Leakage gate (ADR-0017 dec.2c): train_robustness_stress must not overlap any eval OR dev suite
#    (exact + near-duplicate, Jaccard >= 0.7). The stress prep already excluded overlaps; this is the
#    independent re-check across all 5 eval + 3 dev suites. Exits non-zero on overlap -> do NOT train.
gate = subprocess.run(
    ["safestack", "data", "overlap", "--train-split", "train_robustness_stress"],
    capture_output=True, text=True,
)
print(gate.stdout[-2000:])
if gate.returncode != 0:
    print(gate.stderr[-2000:])
    raise SystemExit("LEAKAGE GATE FAILED: train_robustness_stress overlaps an eval/dev suite -- do NOT train.")
print("leakage gate PASS")

train_robustness_stress: 821 train records vs 8 eval suites -> 0 exact, 0 near-dup (>= 0.7)

leakage gate PASS


In [8]:
# 8. Continue-train the five stressed adapters (ADR-0017 dec.3). Each run RESUMES the pinned SFT
#    adapter (init_adapter @ 05266a9b, is_trainable=True) and trains 1 epoch on its budget's slice --
#    the base-mismatch guard runs on load. The long GPU step: five continue-trains (base reload each).
#    Each writes a private adapter + committed loss curves, staged to Drive so a session death after a
#    budget finishes does not force a retrain of it.
import shutil

for b in BUDGETS:
    print(f"=== train stress_mistral_lora_b{b} ===")
    t = subprocess.run(
        ["safestack", "train", "sft", "-c", STRESS_CONFIG[b]],
        capture_output=True, text=True,
    )
    print(t.stdout[-2000:])
    if t.returncode != 0:
        print(t.stderr[-4000:])
        raise SystemExit(f"training failed at budget {b}")
    assert os.path.exists(f"{ADAPTER_LOCAL[b]}/adapter_config.json"), f"adapter b{b} not written"
    shutil.copytree(ADAPTER_LOCAL[b], f"{ADAPTERS}/stress_mistral_lora_b{b}", dirs_exist_ok=True)
    print(f"adapter b{b} at {ADAPTER_LOCAL[b]} (+ staged to Drive)")
print("all", len(BUDGETS), "stressed adapters trained")

=== train stress_mistral_lora_b10 ===
{'loss': '2.876', 'grad_norm': '41.11', 'learning_rate': '0', 'epoch': '1'}
{'train_runtime': '4.912', 'train_samples_per_second': '2.036', 'train_steps_per_second': '0.204', 'train_loss': '2.876', 'epoch': '1'}
trained stress_mistral_lora_b10: 10 train / 0 val, final_train_loss=2.875715970993042, adapter -> adapters/stress_mistral_lora_b10

adapter b10 at adapters/stress_mistral_lora_b10 (+ staged to Drive)
=== train stress_mistral_lora_b50 ===
{'loss': '3.069', 'grad_norm': '41.67', 'learning_rate': '0', 'epoch': '0.3077'}
{'loss': '2.973', 'grad_norm': '42.78', 'learning_rate': '2e-05', 'epoch': '0.6154'}
{'loss': '1.522', 'grad_norm': '24.09', 'learning_rate': '1.5e-05', 'epoch': '0.9231'}
{'loss': '0.9864', 'grad_norm': '16.79', 'learning_rate': '5e-06', 'epoch': '1'}
{'train_runtime': '10.64', 'train_samples_per_second': '4.7', 'train_steps_per_second': '0.376', 'train_loss': '2.138', 'epoch': '1'}
trained stress_mistral_lora_b50: 50 train / 

In [9]:
# 9. Upload each stressed adapter to its OWN PRIVATE HF-Hub repo for an immutable id (ADR-0017 dec.7 as
#    amended by Amendment 2). The weights live here, NEVER in the public git repo and NEVER as a public
#    model. Each returned commit SHA is the adapter_revision to pin into that budget's card (FU4).
from huggingface_hub import HfApi, create_repo

STRESS_REPO = {b: f"kambleakash0/safestack-stress-mistral-lora-b{b}" for b in BUDGETS}
STRESS_SHA = {}
api = HfApi()
for b in BUDGETS:
    create_repo(STRESS_REPO[b], private=True, repo_type="model", exist_ok=True,
                token=os.environ["HF_TOKEN"])
    # Fail loud if the repo somehow already exists PUBLIC -- create_repo(exist_ok=True) does NOT flip an
    # existing repo's visibility, and a stressed adapter must never land in a public repo (never public).
    if api.model_info(STRESS_REPO[b], token=os.environ["HF_TOKEN"]).private is not True:
        raise SystemExit(f"{STRESS_REPO[b]} is not private -- refusing to upload a stressed adapter")
    commit = api.upload_folder(
        repo_id=STRESS_REPO[b],
        folder_path=ADAPTER_LOCAL[b],
        repo_type="model",
        commit_message=f"robustness-stress LoRA adapter (stress_mistral_lora_b{b})",
        token=os.environ["HF_TOKEN"],
    )
    STRESS_SHA[b] = getattr(commit, "oid", None) or api.model_info(
        STRESS_REPO[b], token=os.environ["HF_TOKEN"]
    ).sha
    print(f"uploaded {STRESS_REPO[b]} @ {STRESS_SHA[b]} (private)")
print("\n-> pin these SHAs as adapter_revision in the FU4 stressed policy cards:")
for b in BUDGETS:
    print(f"   stress_mistral_lora_b{b}: adapter={STRESS_REPO[b]}  adapter_revision={STRESS_SHA[b]}")

uploaded kambleakash0/safestack-stress-mistral-lora-b10 @ 4a252a4be8f1294c42562b19674d14e2c7004a82 (private)
uploaded kambleakash0/safestack-stress-mistral-lora-b50 @ fa47940ce11cf9bd80fee84a2f3e089a1b75543e (private)
uploaded kambleakash0/safestack-stress-mistral-lora-b100 @ ae71926c4e8e1ff12949e3f045587644f08037bb (private)
uploaded kambleakash0/safestack-stress-mistral-lora-b250 @ eb845b9dc37f2497a7784c2778b5b499c0eb0a0e (private)
uploaded kambleakash0/safestack-stress-mistral-lora-b411 @ 8df336bcc553db4e129b1e0d639f41ea43c8d98c (private)

-> pin these SHAs as adapter_revision in the FU4 stressed policy cards:
   stress_mistral_lora_b10: adapter=kambleakash0/safestack-stress-mistral-lora-b10  adapter_revision=4a252a4be8f1294c42562b19674d14e2c7004a82
   stress_mistral_lora_b50: adapter=kambleakash0/safestack-stress-mistral-lora-b50  adapter_revision=fa47940ce11cf9bd80fee84a2f3e089a1b75543e
   stress_mistral_lora_b100: adapter=kambleakash0/safestack-stress-mistral-lora-b100  adapter_r

In [10]:
# 10. Loss curves (committed, aggregate-only): the tracked train loss per budget. val_fraction is 0
#     (dose-exact: every example is trained on, none held out), so n_val is 0 and there is no val curve.
import json

for b in BUDGETS:
    c = json.load(open(f"reports/train_curves/stress_mistral_lora_b{b}.json"))
    hp = c["hyperparameters"]
    print(f"b{b}: split={c['train_split']} n_train={c['n_train']} n_val={c['n_val']} "
          f"precision={hp['precision']} final_train_loss={c['final_train_loss']} "
          f"train_points={len(c['curves']['train'])}")

b10: split=train_robustness_stress n_train=10 n_val=0 precision=bf16 final_train_loss=2.875715970993042 train_points=1
b50: split=train_robustness_stress n_train=50 n_val=0 precision=bf16 final_train_loss=0.9863647222518921 train_points=4
b100: split=train_robustness_stress n_train=100 n_val=0 precision=bf16 final_train_loss=0.3631631135940552 train_points=7
b250: split=train_robustness_stress n_train=250 n_val=0 precision=bf16 final_train_loss=0.00035458087222650647 train_points=16
b411: split=train_robustness_stress n_train=411 n_val=0 precision=bf16 final_train_loss=1.4267547157942317e-05 train_points=26


## After the run

**Commit (aggregate-only)** from the repo, then push:
- `reports/train_curves/stress_mistral_lora_b{10,50,100,250,411}.json` — the five loss curves
- this executed notebook (aggregate-only outputs; verify no raw prompts / generations / targets appear)

**Private, never committed, never public (dec.7 as amended by Amendment 2):** the five stressed adapter
weights (their **private** HF-Hub repos + `adapters/`) and the raw stress prompts + affirmative-onset
targets. Do not make the repos or the Drive copies public.

**Pin the stressed cards (FU4):** in the FU4 PR create `configs/models/stress_mistral_lora_b{budget}.yaml`
(one per budget) with `adapter:` = the printed **private** repo and `adapter_revision:` = the printed
commit SHA (ADR-0017 Amendment 2), so each C9/C10(b) has one immutable policy identity (ADR-0015
dec.6/7b; ADR-0017 dec.4). Everything else in the card matches the C5 card (bf16 base, no quantization)
so budget-0 = C5 anchors the dose-response.

**Next (FU4):** run the b\* selection on the three DEV suites (dec.4 — the largest budget still passing
the mode-collapse tripwire), then C9(b\*) real generation + C10(b\*) cache-hit, and record H4/H5 as
ADR-0018. Never read b\* or C9/C10 on the locked test before that selection (rule 3).